In [2]:
import pandas as pd
import numpy as np
import holidays

# Load the pristine dataset
df = pd.read_csv('ml_ready_bookings.csv')
df['arrival_date'] = pd.to_datetime(df['arrival_date'])

print(f"Total rows: {df.shape[0]}")
print(df.info())

Total rows: 119386
<class 'pandas.DataFrame'>
RangeIndex: 119386 entries, 0 to 119385
Data columns (total 39 columns):
 #   Column                                 Non-Null Count   Dtype         
---  ------                                 --------------   -----         
 0   hotel                                  119386 non-null  str           
 1   lead_time                              119386 non-null  int64         
 2   booking_date                           119386 non-null  str           
 3   arrival_date                           119386 non-null  datetime64[us]
 4   stays_in_weekend_nights                119386 non-null  int64         
 5   stays_in_week_nights                   119386 non-null  int64         
 6   adults                                 119386 non-null  int64         
 7   children                               119386 non-null  int64         
 8   babies                                 119386 non-null  int64         
 9   country                                1

In [3]:
# 1. Convert booking_date to datetime
df['booking_date'] = pd.to_datetime(df['booking_date'])

# 2. Convert ID columns to Nullable Integers (capital 'I' Int64)
df['agent'] = df['agent'].astype('Int64')
df['company'] = df['company'].astype('Int64')

# 3. Double-check the clean types
df[['booking_date', 'agent', 'company']].info()

<class 'pandas.DataFrame'>
RangeIndex: 119386 entries, 0 to 119385
Data columns (total 3 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   booking_date  119386 non-null  datetime64[us]
 1   agent         103048 non-null  Int64         
 2   company       6797 non-null    Int64         
dtypes: Int64(2), datetime64[us](1)
memory usage: 3.0 MB


In [4]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 119386 entries, 0 to 119385
Data columns (total 39 columns):
 #   Column                                 Non-Null Count   Dtype         
---  ------                                 --------------   -----         
 0   hotel                                  119386 non-null  str           
 1   lead_time                              119386 non-null  int64         
 2   booking_date                           119386 non-null  datetime64[us]
 3   arrival_date                           119386 non-null  datetime64[us]
 4   stays_in_weekend_nights                119386 non-null  int64         
 5   stays_in_week_nights                   119386 non-null  int64         
 6   adults                                 119386 non-null  int64         
 7   children                               119386 non-null  int64         
 8   babies                                 119386 non-null  int64         
 9   country                                118898 non-null  str

In [5]:
# I've noticed some discrepacies and problematics in the datatypes, so I decided to sped some time fixing them.# Downcast monetary columns to float32 to save memory
df['adr'] = df['adr'].astype('float32')
df['total_hotel_revenue_on_arrival_date'] = df['total_hotel_revenue_on_arrival_date'].astype('float32')

# Let's check the memory usage now!
print(f"New Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

New Memory Usage: 77.86 MB


In [6]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 119386 entries, 0 to 119385
Data columns (total 39 columns):
 #   Column                                 Non-Null Count   Dtype         
---  ------                                 --------------   -----         
 0   hotel                                  119386 non-null  str           
 1   lead_time                              119386 non-null  int64         
 2   booking_date                           119386 non-null  datetime64[us]
 3   arrival_date                           119386 non-null  datetime64[us]
 4   stays_in_weekend_nights                119386 non-null  int64         
 5   stays_in_week_nights                   119386 non-null  int64         
 6   adults                                 119386 non-null  int64         
 7   children                               119386 non-null  int64         
 8   babies                                 119386 non-null  int64         
 9   country                                118898 non-null  str

In [7]:
git add .gitignore
git add notebooks/04_holiday_and_event_engineering.ipynb
git commit -m "feat: initialize notebook 04 for holiday and event engineering"
git push origin main

SyntaxError: invalid decimal literal (888513273.py, line 2)

In [1]:
import pandas as pd
import numpy as np
import holidays

# 1. Load the pristine dataset from the local folder
df = pd.read_csv('ml_ready_bookings.csv')

# 2. Correct the Date Data Types
df['arrival_date'] = pd.to_datetime(df['arrival_date'])
df['booking_date'] = pd.to_datetime(df['booking_date'])

# 3. Clean up ID columns (using nullable Int64 so NaNs don't force float64)
df['agent'] = df['agent'].astype('Int64')
df['company'] = df['company'].astype('Int64')

# 4. Optimize Money Columns (downcast to float32 to save memory)
df['adr'] = df['adr'].astype('float32')
df['total_hotel_revenue_on_arrival_date'] = df['total_hotel_revenue_on_arrival_date'].astype('float32')

# 5. Initialize Portuguese Holiday Calendar
pt_holidays = holidays.Portugal(years=range(df['arrival_date'].dt.year.min(), df['arrival_date'].dt.year.max() + 1))

# 6. Engineer Holiday & Long Weekend Features
df['is_holiday'] = df['arrival_date'].apply(lambda d: 1 if d in pt_holidays else 0)
df['holiday_name'] = df['arrival_date'].apply(lambda d: pt_holidays.get(d) if d in pt_holidays else "None")

# A weekend is a long weekend if it's a holiday AND falls on Fri, Sat, Sun, or Mon
df['is_long_weekend'] = np.where(
    (df['is_holiday'] == 1) & 
    (df['arrival_date'].dt.dayofweek.isin([0, 4, 5, 6])), 
    1, 
    0
)

# 7. Print Audit Summary
print("--- ENVIRONMENT & DATA AUDIT ---")
print(f"Total bookings loaded: {df.shape[0]:,}")
print(f"Memory footprint: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\n--- NEW HOLIDAY FEATURES SNEAK PEEK ---")
print(df[df['is_holiday'] == 1][['arrival_date', 'holiday_name', 'is_long_weekend']].drop_duplicates().head(5))

--- ENVIRONMENT & DATA AUDIT ---
Total bookings loaded: 119,386
Memory footprint: 85.81 MB

--- NEW HOLIDAY FEATURES SNEAK PEEK ---
    arrival_date                                       holiday_name  \
44    2016-05-01                                 Dia do Trabalhador   
135   2016-05-26                                      Corpo de Deus   
207   2017-04-16                                             Páscoa   
216   2017-06-15                                      Corpo de Deus   
220   2017-06-10  Dia de Portugal, de Camões e das Comunidades P...   

     is_long_weekend  
44                 1  
135                0  
207                1  
216                0  
220                1  


In [2]:
git add .gitignore
git add phyton_notebooks/04_holiday_and_event_engineering.ipynb
git commit -m "feat: initialize holiday engineering notebook and ignore CSVs"
git push origin main

SyntaxError: invalid decimal literal (2016495902.py, line 2)